In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd 
from selenium.webdriver.chrome.options import Options
from selenium import webdriver
import random
import re
import time
import folium

## Busca as Linhas de Onibus

In [ ]:
url = 'https://quantotempofalta.piracicabana.com.br/pg_FindLines.php'

In [ ]:
# Function to get webpage code using selenium
def scrape_url(url: str):
        user_agents = [
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
                , "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36"
                , "Mozilla/5.0 (iPhone; CPU iPhone OS 14_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0 Mobile/15E148 Safari/604.1"
                , "Mozilla/5.0 (Linux; Android 10; SM-G960F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.93 Mobile Safari/537.36"
                ]

        chrome_options = Options()
        chrome_options.add_argument('--headless')  # Executar em modo headless, sem abrir uma janela do navegador
        chrome_options.add_argument('--disable-gpu')  # Desabilitar GPU para melhorar o desempenho em modo headless
        chrome_options.add_argument(f"--user-agent={random.choice(user_agents)}")

        chrome_options.add_experimental_option("prefs", {
            "download.default_directory": "/tmp",  # Define o diretório de download como /tmp para evitar downloads
            "download.prompt_for_download": False,  # Desabilita a janela de diálogo de download
            "download.directory_upgrade": True,
            "safebrowsing.enabled": True
        })



        driver = webdriver.Chrome(options=chrome_options)
        driver.set_page_load_timeout(60)

        driver.get(url)
        html = driver.page_source
        driver.quit()
        return html 

In [ ]:
response = scrape_url(url)

In [ ]:
soup = BeautifulSoup(response, 'lxml')

In [ ]:
middle_div = soup.find('div', id='middle')

In [ ]:
rows = []
for info in middle_div.find_all('a', class_='select_lines'):
    link = info.get('href')

    nome_linha = info.find('strong').text.strip()

    # descrição
    spans = info.find_all('span')
    descricao = spans[1].text.strip() if len(spans) > 1 else None

    rows.append({
        'linha': nome_linha,
        'descricao': descricao,
        'link': link
    })

In [ ]:
df_linha = pd.DataFrame(rows)

In [ ]:
df_linha.to_parquet('df_linhas_onibus_santos.parquet')

## Busca as Coordenas das Rotas de Ida e Volta

In [ ]:
#re.findall(r"latlng\w+", script_text)

In [ ]:
df_linha = pd.read_parquet('df_linhas_onibus_santos.parquet')

In [ ]:
def parse_exibe_pontos(script_text):

    pattern = r"ExibePontosLinha\(\s*([-\d\.]+)\s*,\s*([-\d\.]+)\s*,\s*'([^']+)'\s*,"

    matches = re.findall(pattern, script_text)

    df = pd.DataFrame(matches, columns=["latitude", "longitude", "texto"])

    df["latitude"] = df["latitude"].astype(float)
    df["longitude"] = df["longitude"].astype(float)

    df["id"] = range(1, len(df) + 1)

    return df[["id", "latitude", "longitude", "texto"]]

def extract_route_block(script_text, name):
    
    pattern = rf"var\s+{name}\s*=\s*\[(.*?)\];"
    
    match = re.search(pattern, script_text, re.S)
    
    if match:
        return match.group(1)
    else:
        return None

def parse_coords(block):

    coords = re.findall(r"lat:([-\d\.]+),\s*lng:([-\d\.]+)", block)

    import pandas as pd

    df = pd.DataFrame(coords, columns=["lat","lng"])
    df["lat"] = df["lat"].astype(float)
    df["lng"] = df["lng"].astype(float)

    return df

In [ ]:
url_descricao = 'https://quantotempofalta.piracicabana.com.br/{parametro}'

df_rotas = pd.DataFrame()
df_descricao_rotas = pd.DataFrame()

for idx, row in df_linha.iterrows():
    try:

        url_rota = url_descricao.format(parametro=row['link'])
        
        get_details = scrape_url(url_rota)

        soup = BeautifulSoup(get_details, 'lxml')

        scripts = soup.find_all("script")
        script_text = " ".join([s.get_text() for s in scripts])

        df_ida = extract_route_block(script_text, "latlngIda")
        df_volta = extract_route_block(script_text, "latlngVolta")
        df_ida = parse_coords(df_ida)
        df_ida["sentido"] = "ida"
        df_volta = parse_coords(df_volta)
        df_volta["sentido"] = "volta"

        df_rota_linha = pd.concat([df_ida, df_volta])
        df_rota_linha['linha'] = row['linha']

        df_rotas = pd.concat([df_rotas, df_rota_linha])

        df_pontos_desc = parse_exibe_pontos(script_text)
        df_pontos_desc['linha'] = row['linha']

        df_descricao_rotas = pd.concat([df_descricao_rotas, df_pontos_desc ])
        

        print(f"Sucesso ao executar {row['linha']}")
        time.sleep(3)
    except Exception as e:
        print(f"Erro ao executar {row['linha']} :: Erro: {e}")
        continue


In [ ]:
df_descricao_rotas.to_parquet('df_pontos_onibus_descricao_santos.parquet')
#df_rotas.to_parquet('df_rotas_onibus_santos.parquet')

In [ ]:
df_descricao_rotas

In [ ]:
df_descricao_rotas[ df_descricao_rotas['linha'] == 'CIRCULAR 013' ]

In [ ]:
df_rotas[ (df_rotas['linha'] == 'CIRCULAR 013') & (df_rotas['sentido'] == 'ida') ]

In [4]:
df_descricao_rotas = pd.read_parquet('df_pontos_onibus_descricao_santos.parquet')
df_rotas = pd.read_parquet('df_rotas_onibus_santos.parquet')
df_linha = pd.read_parquet('df_linhas_onibus_santos.parquet')

In [6]:
mapa = folium.Map(location=[-23.9545155,-46.309058], zoom_start=13, tiles='CartoDB positron')


rota_ida = df_rotas[ (df_rotas['linha'] == 'CIRCULAR 013') & (df_rotas['sentido'] == 'ida') ][['lat','lng']].values.tolist()
folium.PolyLine(rota_ida,color="blue",weight=4).add_to(mapa)
rota_ida = df_rotas[ (df_rotas['linha'] == 'CIRCULAR 013') & (df_rotas['sentido'] == 'volta') ][['lat','lng']].values.tolist()
folium.PolyLine(rota_ida,color="green",weight=4).add_to(mapa)


#rota_ida = df_rotas[ (df_rotas['linha'] == 'CIRCULAR 013') & (df_rotas['sentido'] == 'ida') ][['lat','lng']].values.tolist()
#rota_volta = df_rotas[ (df_rotas['linha'] == 'CIRCULAR 013') & (df_rotas['sentido'] == 'volta') ][['lat','lng']].values.tolist()


#folium.PolyLine(rota_ida,color="blue",weight=4).add_to(mapa)
#folium.PolyLine(rota_volta,color="green",weight=4).add_to(mapa)



#for idx, row in df_descricao_rotas[ df_descricao_rotas['linha'] == 'CIRCULAR 013' ].iterrows():

    #folium.Marker([row['latitude'], row['longitude']]).add_to(mapa)

mapa
